# 06 — Spell Damage

Primary spell casting showcase — single-spell execution, fizzle analysis,
resist analysis, elemental protection sweeps, and DPS from casting delay.

In [ ]:
import logging
from pathlib import Path

from omega.config.spells import Spell
from omega.model.constants import (
    SKILLID_EVALINT, SKILLID_MAGERY, SKILLID_MAGICRESISTANCE,
    SKILLID_MEDITATION, SKILLID_WRESTLING,
)
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, SpellParameterSweep, SpellScenario, Variable,
    run_spell_scenario, run_spell_sweep,
)
from omega.reporting.tables import comparison_table, summary_table, format_table_html
from omega.reporting.plots import (
    damage_histogram, damage_breakdown, damage_vs_parameter,
    fizzle_rate_vs_parameter, spell_comparison,
)
from omega.logging import setup_logging
from IPython.display import HTML
import dataclasses

setup_logging(level=logging.ERROR)

SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

# --- Shared combatants ---
MAGE = CombatantSpec(
    name="Mage",
    skills={SKILLID_MAGERY: 100, SKILLID_EVALINT: 100, SKILLID_MEDITATION: 100},
    str_=50, dex_=50, int_=120,
    class_levels={"IsMage": 5},
)
TARGET = CombatantSpec(
    name="Target Dummy", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    armor=ArmorSpec(ar=30),
)
ITERATIONS = 200
BASE_SEED = 42

print("Setup complete.")

## 1. Single Spell — Fireball

Run a single Fireball scenario and inspect the damage distribution.

In [ ]:
fireball = SpellScenario(
    caster=MAGE, target=TARGET,
    spell_id=Spell.FIREBALL,
    iterations=ITERATIONS, base_seed=BASE_SEED,
    npc_mode=False,
)

fb_result = run_spell_scenario(fireball, shard=shard)

print(f"Spell: Fireball (Circle {fb_result.raw_results[0].circle})")
print(f"Fizzle rate:  {fb_result.ratios.fizzle_rate:.1%}")
print(f"Resist rate:  {fb_result.ratios.resist_rate:.1%}")
print(f"Cast rate:    {fb_result.ratios.hit_rate:.1%}")
print(f"Mean damage (overall):  {fb_result.damage_stats.mean:.2f}")
print(f"Mean damage (on cast):  {fb_result.damage_stats_on_cast.mean:.2f}")
print(f"Range: {fb_result.damage_stats.min:.0f} – {fb_result.damage_stats.max:.0f}")

In [ ]:
damage_histogram(fb_result, title="Fireball — 200 casts")

In [ ]:
damage_breakdown(fb_result, title="Fireball — Damage Breakdown")

## 2. Fizzle Analysis — Low vs High Magery

Fizzle rate is controlled by `CheckSkill(caster, MAGERY, circle * 10)` —
low Magery fizzles most casts, while high Magery almost never fizzles.

In [ ]:
fizzle_results = {}
for magery_val in [30, 60, 100]:
    caster = dataclasses.replace(MAGE, skills={SKILLID_MAGERY: magery_val, SKILLID_EVALINT: 100})
    cell = run_spell_scenario(
        SpellScenario(caster=caster, target=TARGET, spell_id=Spell.FIREBALL,
                      iterations=ITERATIONS, base_seed=BASE_SEED, npc_mode=False),
        shard=shard,
    )
    label = f"Magery {magery_val}"
    fizzle_results[label] = cell
    print(f"  {label}  fizzle={cell.ratios.fizzle_rate:.0%}  mean={cell.damage_stats.mean:.2f}")

In [ ]:
spell_comparison(fizzle_results, title="Fireball: Low vs High Magery")

In [ ]:
rows = comparison_table(
    fizzle_results,
    stats=["mean", "mean_on_cast", "median", "p5", "p95",
           "fizzle_rate", "resist_rate", "cast_rate"],
)
HTML(format_table_html(rows))

## 3. Resist Analysis — MagicResistance Sweep

Defender MagicResistance affects the `Resisted()` check — higher resist means
the spell's damage is halved more often.

In [ ]:
resist_sweep = SpellParameterSweep(
    scenario=SpellScenario(
        caster=MAGE, target=dataclasses.replace(
            TARGET, skills={SKILLID_MAGICRESISTANCE: 0},
        ),
        spell_id=Spell.FIREBALL,
        iterations=100, base_seed=1234, npc_mode=False,
    ),
    variables=(
        Variable.from_range("target", f"skills.{SKILLID_MAGICRESISTANCE}",
                            start=0, stop=130, step=10),
    ),
)

resist_result = run_spell_sweep(resist_sweep, shard=shard)
print(f"{len(resist_result.cells)} cells completed in {resist_result.total_time:.1f}s")

In [ ]:
damage_vs_parameter(
    resist_result,
    f"target.skills.{SKILLID_MAGICRESISTANCE}",
    title="Fireball: Mean Damage vs Target Resist",
)

In [ ]:
fizzle_rate_vs_parameter(
    resist_result,
    f"target.skills.{SKILLID_MAGICRESISTANCE}",
    title="Fireball: Fizzle & Resist Rate vs Target Resist",
)

## 4. Elemental Protection Sweep

Fireball is fire-element damage. Sweeping the target's `FireProtection`
from 0 to 100 shows how elemental protection reduces spell damage.

In [ ]:
prot_sweep = SpellParameterSweep(
    scenario=SpellScenario(
        caster=MAGE, target=TARGET,
        spell_id=Spell.FIREBALL,
        iterations=100, base_seed=1234, npc_mode=False,
    ),
    variables=(
        Variable.from_range("target", "properties.FireProtection",
                            start=0, stop=100, step=10),
    ),
)

prot_result = run_spell_sweep(prot_sweep, shard=shard)
print(f"{len(prot_result.cells)} cells completed in {prot_result.total_time:.1f}s")

In [ ]:
damage_vs_parameter(
    prot_result,
    "target.properties.FireProtection",
    title="Fireball: Mean Damage vs Fire Protection",
)

In [ ]:
rows = summary_table(
    prot_result,
    stats=["mean", "mean_on_cast", "median", "p5", "p95",
           "fizzle_rate", "resist_rate", "elem_total_net"],
)
HTML(format_table_html(rows))

## 5. PvP Scaling

Player vs Player spell damage includes a 60% reduction via `ApplyTheDamage()`,
same as weapon hits. Compare PvP vs PvE (NPC target) damage.

In [ ]:
pvp_target = dataclasses.replace(TARGET, is_npc=False, name="Player Target")

pvp_results = {}
for label, target in [("vs NPC", TARGET), ("vs Player", pvp_target)]:
    cell = run_spell_scenario(
        SpellScenario(caster=MAGE, target=target, spell_id=Spell.FIREBALL,
                      iterations=ITERATIONS, base_seed=BASE_SEED, npc_mode=False),
        shard=shard,
    )
    pvp_results[label] = cell
    print(f"  {label:12s}  mean={cell.damage_stats.mean:.2f}  "
          f"on_cast={cell.damage_stats_on_cast.mean:.2f}")

In [ ]:
rows = comparison_table(
    pvp_results,
    stats=["mean", "mean_on_cast", "median", "min", "max",
           "fizzle_rate", "resist_rate", "effective_dps"],
)
HTML(format_table_html(rows))

## 6. Spell Pipeline Metrics

Each `SpellResult` carries a `metrics` dict populated by `__RecordSimulatorMetric()`
calls in the shard scripts — providing visibility into the damage pipeline.

In [ ]:
# Run a small batch and inspect raw metrics
inspect_result = run_spell_scenario(
    SpellScenario(caster=MAGE, target=TARGET, spell_id=Spell.FIREBALL,
                  iterations=5, base_seed=99, npc_mode=False),
    shard=shard,
)

for i, sr in enumerate(inspect_result.raw_results):
    print(f"--- Cast {i + 1} ---")
    print(f"  fizzled={sr.fizzled}  resisted={sr.resisted}  "
          f"final_damage={sr.final_damage:.1f}")
    print(f"  metrics keys: {sorted(sr.metrics.keys())}")
    for key in ["spell_dice_roll", "spell_base_damage", "spell_efficiency_penalty"]:
        if key in sr.metrics:
            print(f"    {key} = {sr.metrics[key]}")
    elem = sr.metrics.get("elemental_applied", [])
    if elem:
        print(f"    elemental_applied: {elem}")
    print()

## 7. DPS from Casting Delay

Spell casting has a virtual casting delay (from `circles.cfg`). Combined with
fizzle rate, this gives **effective DPS** — damage per second accounting for
both casting time and fizzle probability.

In [ ]:
ts = fb_result.timing
if ts:
    print(f"Casting delay:  {ts.swing_delay_ms:.0f}ms")
    print(f"Casts/sec:      {ts.swings_per_second:.3f}")
    print(f"DPS (mean):     {ts.dps_mean:.1f}")
    print(f"DPS (on cast):  {ts.dps_on_hit:.1f}")
    print(f"Effective DPS:  {ts.effective_dps:.1f}")

from omega.simulation.stats import SimulationResult
rows = summary_table(
    SimulationResult(cells=[fb_result]),
    stats=["mean", "mean_on_cast", "fizzle_rate", "resist_rate",
           "swing_delay_ms", "swings_per_sec", "dps_mean", "effective_dps"],
)
HTML(format_table_html(rows))